# OSU AI Club Poker Competition 2026
*A Guide to Building and Submitting Your Poker Agent*

---

### Table of Contents

1. [Installation Guide](#installation-guide)
2. [Poker Environment Overview](#poker-environment-overview)
3. [Create an Agent](#create-an-agent)
4. [Test Your Agent](#test-your-agent)
5. [Competition Rules & Submission](#competition-rules-and-rewards)


## Installation Guide

Before you can start creating an agent, you must install the necessary dependicies. From the project root directory, run:

```
pip install -r requirements.txt
```

**Optional:** For a clean environment, consider using a virtual environment first:

```
python -m venv venv
```

Then activate it (e.g., `venv\Scripts\activate` on Windows, `source venv/bin/activate` on macOS/Linux) and run the `pip install` command above.

## Poker Environment Overview

This competition utilizes a custom implementation of Petting Zoo's [Texas Hold'em No Limit Environment](https://pettingzoo.farama.org/environments/classic/texas_holdem_no_limit/).

### Observation Space

In this custom implementation, the observation is a dictionary containing three elements: the original `observation` and `action_mask` included in Petting Zoo's Environment, and a third `human_readable` element designed to support rule-based bots.

#### 1. The Reinforcement Learning Vector
> **Key:** `observation['observation']`

This element matches the original Petting Zoo implementation and is ideal for **Reinforcement Learning** models.
* **Structure:** A 54-length vector.
* **Content:** The first 52 entries represent the cards in the player's hand and community cards.
    * `1`: The card is present (in hand or on board).
    * `0`: The card is not present.

![rl-observation-space](<images/rl-observation-space.png>)
---

#### 2. Human-Readable State
> **Key:** `observation['human_readable']`

This is a custom dictionary designed for **rule-based decision making** and debugging. It translates the raw vector into explicit values (e.g., current pot, specific card ranks, stack sizes).

![human-readable-observation-space](<images/human-readable-observation-space.png>)
---

#### 3. Action Mask
> **Key:** `observation['action_mask']`

This element tells the agent what moves it can make, given the gamestate.
* **Structure:** A binary vector with **5 entries**.
* **Function:** Each entry corresponds to a specific action.
    * `1` = Action is **Legal**.
    * `0` = Action is **Illegal**.

![action-mask](<images/action-space.png>)

### Custom Variant

This competition uses a **Weak Hand Multiplier** that rewards aggressive play with bad starting hands and penalizes losing to them.

**Reward multipliers:**
- **Winning with a weak hand:** Your chip reward is **doubled** when you win a hand and your hole cards qualify as a weak starting hand.
- **Losing to a weak hand:** Your chip loss is **doubled** when you lose to an opponent whose hole cards qualified as a weak starting hand.

**What constitutes a weak hand?**

Weak hands are judged by your **hole cards only** (the first 2 cards you get). A hand qualifies as weak if it meets all of these criteria:

1. Not a pair 
2. Not suited 
3. Both cards below 10
4. Not a connector (cards not one rank apart) 

Examples: 9-7 offsuit, 8-4 offsuit, 6-3 offsuit.

The observation includes `human_readable['is_weak_hand']`, which is `True` when your current hole cards qualify for the bonus if you win.


## Create an Agent

An agent is a simple Python class responsible for making decisions. It must implement a method called `act` which receives the current game state (`observation`). Your goal is to use the observation data to return a valid integer action.

### Agent Structure

You will design a Python class with the following structure:

In [ ]:
import numpy as np 
from typing import Dict
from poker_env import Action

class AlwaysFoldAgent:
    def act(self, observation: Dict) -> int:
        
        # Get the action mask (binary vector of valid moves)
        # Example: action_mask = [1, 0, 1, 1, 0], indicates actions 0, 2, and 3 are valid.
        action_mask = observation["action_mask"]

        # Get the integer indices of all legal actions
        # Example: turns [1, 0, 1, 1, 0] -> [0, 2, 3]
        valid_actions = np.flatnonzero(action_mask)  

        # Always try to Fold if it's legal.
        # If not, pick the first available legal action  
        # NOTE: Action.FOLD is an ENUM and equivalent to 0 
        if Action.FOLD in valid_actions:
            return Action.FOLD
        return valid_actions[0]

### Tips for Creating A Deep/Reinforcement Learning Agent

You may want to train a **reinforcement learning** agent for this competition, which is great! This competition is set up to make that as easy as possible. Here are some things to keep in mind.

**The multi-agent problem.** Poker is a 2-player game using PettingZoo's AEC API, but most RL libraries expect a single-agent Gymnasium environment. To bridge this gap, you need a wrapper that auto-plays the opponent's turns so your agent sees a standard `gym.Env`. The provided `rl_example/gym_wrapper.py` shows one way to do this.

**Action masking.** Not all possible actions are legal every turn. Your agent must respect `observation['action_mask']` and only choose legal actions. Some algorithms handle this natively (e.g. MaskablePPO), others will require you to manually mask illegal outputs.

**Observation design.** You have two built-in observation options:
* The **raw 54-dim vector** (`observation['observation']`) — binary card presence, sparse and harder to learn from.
* The **human-readable state** (`observation['human_readable']`) — pot, stacks, hand, community cards, round, etc.

You can feed either directly to a network, or engineer your own feature vector. The provided `rl_example/feature_extractor.py` builds a 24-dim normalized vector as one example, but experimenting with different representations can make a big difference.

**The provided example.** The `rl_example/` directory contains a working but basic PPO agent. It is meant as a **starting point**, not a ceiling. To try it:
1. Train the agent by running: `python -m rl_example.train` (saves to `models/`)
2. Use the trained agent: import `DRLAgent` from `rl_example.drl_agent` in `test_agent.py` and use it like any other agent.

There is plenty of room for improvement — better features, different algorithms, smarter training strategies, and hyperparameter tuning can all yield stronger agents.

**Resources:** The following resources may be useful when creating your first RL agent.
TODO UPDATE THIS SECTION
* [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) — RL algorithm library
* [sb3-contrib (MaskablePPO)](https://sb3-contrib.readthedocs.io/en/master/modules/ppo_mask.html) — action-masked PPO
* [Gymnasium](https://gymnasium.farama.org/) — single-agent env API
* [PettingZoo](https://pettingzoo.farama.org/) — multi-agent env API
* [Spinning Up in Deep RL](https://spinningup.openai.com/) — RL fundamentals

## Test Your Agent

Once you have implemented your agent's logic, validate its performance by running:

```
python test_agent.py
```

By default, this simulates 10,000 hands of your `MyAgent` vs `HeuristicAgent` and prints win rates and average chip gains.

To test your agent against different opponents, open `test_agent.py` and update which agents are passed into `run_competition`.

In [ ]:
from poker_env import TexasHoldEm
from agents import RandomAgent, HeuristicAgent
from agent import MyAgent

def run_competition(num_hands, agent_a, agent_b, seed=None):
    # ... see test_agent.py for full implementation details
    pass

if __name__ == "__main__":
    # Edit this line here to change which agents you want to test
    run_competition(10_000, MyAgent(), HeuristicAgent())

## Competition Rules and Rewards

TODO: THESE RULES AND REWARDS NEED TO BE DISCUSSED AND UPDATED AND ALL CURRENT RULES ARE SIMPLY PLACEHOLDERS. NOT SURE ABOUT TIME/MEM REQUIREMENTS, USE OF OUTSIDE RESOURCES (IDEALLY PEOPLE CAN'T JUST COPY PASTE SOMEONE ELSE'S CODE), AND PRETTY MUCH ALL OF IT.

### Tournament Structure
The competition will be structured as a **1v1 Tournament**. 

* **Format:** [TODO: ROUND ROBIN / SINGLE ELIMINATION BRACKET]
* **Match Length:** Each match consists of **10,000 hands**.
* **Fairness (Duplicate Poker):** To reduce luck, we will use a "Duplicate Poker" system. 
    * Agents will play 5,000 hands as Player 1 and 5,000 hands as Player 2.
    * The same RNG seeds will be used for both sets, meaning both agents will face the exact same card distribution. The agent that accumulates the most chips across both legs wins the match.

### Submission Requirements
1.  **Language:** All agents must be written in **Python**.
2.  **File Format:** Submit a single file named `agent.py` (or `[TeamName]_agent.py`).
4.  **Dependencies:** You may use standard Python libraries (math, random, etc.) and `numpy`.
    * *Prohibited:* [TODO: NOT SURE IF WE WANT TO PROHIBIT ANY LIBRARIES OR ]

### Agent Constraints & Fair Play
To ensure the tournament runs smoothly, all agents must adhere to the following strict constraints:

1.  **Time Limit:** Your `act` function must return a move within **[0.1] seconds**. Agents exceeding this limit will strictly Fold or be disqualified.
2.  **Memory Limit:** Your agent must not exceed **[500 MB]** of RAM.
3.  **Statelessness:** Agents should not attempt to store data between matches (e.g., writing to disk is prohibited).
4.  **No Cheating:** * Agents must not attempt to access the `env` object's internal variables (e.g., looking at the opponent's cards or the deck).
    * Agents must not attempt to access the internet.
    * Any attempt to "hack" the runner script will result in immediate disqualification.
    * The use of AI is allowed and outside resources are allowed. NOTE: UPDATE THIS LINE. We reccommend not copying code from others?

### Rewards
Prizes will be awarded to the top performing agents:

* 🥇 **1st Place:** [TODO: PRIZE, e.g., $50 Gift Card and AIC Merchandise]
* 🥈 **2nd Place:** [TODO: PRIZE $25 Gift Card]
* 🥉 **3rd Place:** [TODO: PRIZE $25 Gift Card]

---
**How to Submit:**
Please email your `agent.py` file to **sullivk3@oregonstate.edu** by **END OF SPRING Week 10 (June something)**.